# 30-Minute Student Baseline: HPO Without LLM

You are configuring an anomaly detection pipeline. The team wants good quality, but also wants to control training data usage, manual review effort, and robustness to noisy data.

This notebook is the baseline condition. There is no LLM assistant. You will inspect tables of previous HPO runs, choose configurations manually, and briefly justify your choices.

## Columns You Need

The tables use student-friendly column names:

- `quality_score`: predicted quality score. Higher is better, but each task has an allowed target range.
- `training_fraction`: how much training data is used. Lower means less data.
- `review_budget`: how much results reviewing is allowed. Lower means less review effort.
- `corruption_level`: expected noise level. Values are `none`, `mild`, or `strong`.

Quick check example: if a task says `review_budget` must be at most `0.30`, then a row with `review_budget = 0.34` violates that task, even if its `quality_score` is high.

## Tasks

Complete the tasks in this order: `task_1`, then `task_2`, then `task_3`.

| Task | What to do |
|---|---|
| `task_1` | Choose one valid configuration. Quality must be 0.80 to 0.92, training fraction 0.40 to 0.70, review budget at most 0.40, corruption can be any value. |
| `task_2` | Choose one valid configuration. Quality must be 0.78 to 0.91, training fraction at most 0.60, review budget at most 0.30, corruption must be `none` or `mild`. This selection becomes the base for the what-if task. |
| `task_3` | What-if task. Starting from your `task_2` configuration, find another configuration with review budget at least 10% lower while keeping similar performance. Similar means quality can drop by at most 0.01. |

Important: the option tables are mixed. Some rows are valid and some rows violate one or more constraints. You must check the columns before choosing.

In [ ]:
# Install requirements. Run this cell first in Colab.
from pathlib import Path
import subprocess
import sys
import urllib.request

REPO = "sabri-manai/user-study-cf-hpo-xai"
REQUIREMENTS_PATH = Path("requirements.txt")
REQUIREMENTS_URL = f"https://raw.githubusercontent.com/{REPO}/main/requirements.txt"

if not REQUIREMENTS_PATH.exists():
    urllib.request.urlretrieve(REQUIREMENTS_URL, REQUIREMENTS_PATH)

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-r",
    str(REQUIREMENTS_PATH),
])

print("Requirements installed.")

In [ ]:
# Setup. Run this cell after installing requirements.
from pathlib import Path
import json
import time
import urllib.request

import numpy as np
import pandas as pd

DATA_PATH = Path("surrogate_ready_dataset/patchcore_surrogate_dataset_xgb.csv")
# Colab opens only the notebook from GitHub, so this URL lets it fetch the CSV automatically.
DATA_URL = "https://raw.githubusercontent.com/sabri-manai/user-study-cf-hpo-xai/main/surrogate_ready_dataset/patchcore_surrogate_dataset_xgb.csv"
OUTPUT_DIR = Path("student_baseline_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

TARGET = "value"
RAW_DISPLAY_COLUMNS = [
    "run_id",
    "value",
    "params_soft_train_fraction",
    "params_soft_review_budget",
    "params_soft_corruption_level",
    "params_backbone",
    "params_batch_size",
    "params_image_size_key",
    "params_layers_key",
    "params_num_neighbors",
    "params_reduction",
]

COLUMN_LABELS = {
    "value": "quality_score",
    "params_soft_train_fraction": "training_fraction",
    "params_soft_review_budget": "review_budget",
    "params_soft_corruption_level": "corruption_level",
    "params_backbone": "backbone",
    "params_batch_size": "batch_size",
    "params_image_size_key": "image_size",
    "params_layers_key": "layers",
    "params_num_neighbors": "num_neighbors",
    "params_reduction": "reduction",
}

TASKS = {
    "task_1": {
        "quality_range": (0.80, 0.92),
        "train_fraction_range": (0.40, 0.70),
        "review_budget_range": (0.00, 0.40),
        "corruption_allowed": ["none", "mild", "strong"],
    },
    "task_2": {
        "quality_range": (0.78, 0.91),
        "train_fraction_range": (0.20, 0.60),
        "review_budget_range": (0.00, 0.30),
        "corruption_allowed": ["none", "mild"],
    },
}

WHAT_IF_TASK = {
    "source_task": "task_2",
    "review_budget_multiplier": 0.90,
    "quality_tolerance": 0.01,
}

CONFIDENCE_LEVELS = {"low", "medium", "high"}
started_at = time.time()

df = pd.read_csv(DATA_PATH).reset_index().rename(columns={"index": "run_id"})
for col in ["value", "params_soft_train_fraction", "params_soft_review_budget"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df["params_soft_corruption_level"] = df["params_soft_corruption_level"].astype(str)


def student_view(rows):
    return rows[RAW_DISPLAY_COLUMNS].rename(columns=COLUMN_LABELS)


print(f"Loaded {len(df)} previous HPO runs.")
print(f"Quality range in dataset: {df['value'].min():.3f} to {df['value'].max():.3f}")

In [ ]:
# Show mixed option tables for Task 1 and Task 2.
def valid_mask_for(task, dataframe=df):
    q_low, q_high = task["quality_range"]
    tf_low, tf_high = task["train_fraction_range"]
    rb_low, rb_high = task["review_budget_range"]
    allowed = task["corruption_allowed"]

    return (
        dataframe["value"].between(q_low, q_high, inclusive="both")
        & dataframe["params_soft_train_fraction"].between(tf_low, tf_high, inclusive="both")
        & dataframe["params_soft_review_budget"].between(rb_low, rb_high, inclusive="both")
        & dataframe["params_soft_corruption_level"].isin(allowed)
    )


OPTION_RUN_IDS = {
    "task_1": [2738, 2565, 2544, 2449, 2541, 2553, 2572, 2549, 1213, 956, 1098, 615],
    "task_2": [2738, 2553, 2565, 2656, 2549, 2541, 2544, 2548, 1317, 716, 1481],
}


def options_for(task_name):
    option_ids = OPTION_RUN_IDS[task_name]
    options = df[df["run_id"].isin(option_ids)].copy()
    options["display_order"] = options["run_id"].map({run_id: i for i, run_id in enumerate(option_ids)})
    return options.sort_values(
        ["value", "params_soft_review_budget", "params_soft_train_fraction"],
        ascending=[False, True, True],
    ).drop(columns=["display_order"])


OPTIONS = {name: options_for(name) for name in TASKS}

for name, table in OPTIONS.items():
    print(f"\n{name}: mixed option table ({len(table)} rows). Not every row is valid.")
    display(student_view(table))

## Step 1: Choose Your Task 1 Configuration

Edit the next cell first. Choose one `run_id` for `task_1`. Also write a short justification and your confidence: `low`, `medium`, or `high`.

In [ ]:
TASK_1_RUN_ID = 2544

TASK_1_JUSTIFICATION = "Write 1-2 sentences explaining your choice."

TASK_1_CONFIDENCE = "medium"

## Step 2: Choose Your Task 2 Configuration


Now choose one `run_id` for `task_2`. This selection becomes the base for the what-if task.


Also write a short justification and your confidence: `low`, `medium`, or `high`.

In [ ]:
TASK_2_BASE_RUN_ID = 2544

TASK_2_JUSTIFICATION = "Write 1-2 sentences explaining your choice."

TASK_2_CONFIDENCE = "medium"

In [ ]:
# Build the what-if table from your Task 2 base choice.
if TASK_2_BASE_RUN_ID is None:
    raise ValueError("Set TASK_2_BASE_RUN_ID before running this cell.")

base_id = int(TASK_2_BASE_RUN_ID)
base_rows = df[df["run_id"] == base_id]
if base_rows.empty:
    raise ValueError(f"run_id {base_id} does not exist.")

base_row = base_rows.iloc[0]
base_value = float(base_row["value"])
base_review_budget = float(base_row["params_soft_review_budget"])
required_max_review_budget = base_review_budget * WHAT_IF_TASK["review_budget_multiplier"]
minimum_acceptable_value = base_value - WHAT_IF_TASK["quality_tolerance"]

print("Task 3 what-if thresholds")
print(f"Base run_id: {base_id}")
print(f"Base quality: {base_value:.4f}")
print(f"Base review budget: {base_review_budget:.4f}")
print(f"Required review budget: <= {required_max_review_budget:.4f} (10% lower than base)")
print(f"Required quality: >= {minimum_acceptable_value:.4f} (drop no more than 0.01)")

what_if_valid = (
    (df["run_id"] != base_id)
    & (df["params_soft_review_budget"] <= required_max_review_budget)
    & (df["value"] >= minimum_acceptable_value)
)

valid_options = df.loc[what_if_valid].sort_values(
    ["value", "params_soft_review_budget"], ascending=[False, True]
).head(4)

keeps_performance_not_budget = df.loc[
    (df["run_id"] != base_id)
    & (df["value"] >= minimum_acceptable_value)
    & (df["params_soft_review_budget"] > required_max_review_budget)
].sort_values("value", ascending=False).head(4)

lowers_budget_not_performance = df.loc[
    (df["run_id"] != base_id)
    & (df["params_soft_review_budget"] <= required_max_review_budget)
    & (df["value"] < minimum_acceptable_value)
    & (df["value"] >= max(0.0, minimum_acceptable_value - 0.05))
].sort_values("value", ascending=False).head(4)

WHAT_IF_OPTIONS = pd.concat(
    [valid_options, keeps_performance_not_budget, lowers_budget_not_performance],
    ignore_index=True,
).drop_duplicates("run_id")
WHAT_IF_OPTIONS = WHAT_IF_OPTIONS.sort_values(
    ["value", "params_soft_review_budget"], ascending=[False, True]
)

print(f"\ntask_3: mixed what-if option table ({len(WHAT_IF_OPTIONS)} rows). Not every row is valid.")
display(student_view(WHAT_IF_OPTIONS))

## Final Answers


Edit the next cell after you have generated the Task 3 what-if table.


- `task_1` is connected to your Task 1 choice above.

- `task_2` is connected to your Task 2 choice above.

- For `task_3`, choose one `run_id` from the what-if table. If you truly think no suitable alternative exists, set `could_not_find` to `True` and leave `run_id` as `None`.

- For every task, include a short justification and confidence: `low`, `medium`, or `high`.

In [ ]:
ANSWERS = {
    "task_1": {
        "run_id": TASK_1_RUN_ID,
        "justification": TASK_1_JUSTIFICATION,
        "confidence": TASK_1_CONFIDENCE,
    },
    "task_2": {
        "run_id": TASK_2_BASE_RUN_ID,
        "justification": TASK_2_JUSTIFICATION,
        "confidence": TASK_2_CONFIDENCE,
    },
    "task_3": {
        "run_id": None,
        "could_not_find": False,
        "justification": "Write 1-2 sentences explaining whether the review-budget reduction preserves performance.",
        "confidence": "medium",
    },
}

In [ ]:
# Check and save your answers.
def to_jsonable(value):
    if isinstance(value, dict):
        return {str(k): to_jsonable(v) for k, v in value.items()}
    if isinstance(value, list):
        return [to_jsonable(v) for v in value]
    if isinstance(value, tuple):
        return [to_jsonable(v) for v in value]
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.bool_):
        return bool(value)
    return value


def validate_confidence(task_name, confidence):
    confidence = str(confidence).strip().lower()
    if confidence not in CONFIDENCE_LEVELS:
        raise ValueError(f"{task_name}: confidence must be one of {sorted(CONFIDENCE_LEVELS)}")
    return confidence


def check_independent_answer(task_name, answer):
    run_id = answer.get("run_id")
    if run_id is None:
        raise ValueError(f"{task_name}: please enter a run_id")

    run_id = int(run_id)
    if run_id not in set(df["run_id"]):
        raise ValueError(f"{task_name}: run_id {run_id} does not exist")

    row = df.loc[df["run_id"] == run_id].iloc[0]
    task = TASKS[task_name]
    q_low, q_high = task["quality_range"]
    tf_low, tf_high = task["train_fraction_range"]
    rb_low, rb_high = task["review_budget_range"]
    allowed = task["corruption_allowed"]

    checks = {
        "quality_ok": q_low <= float(row["value"]) <= q_high,
        "train_fraction_ok": tf_low <= float(row["params_soft_train_fraction"]) <= tf_high,
        "review_budget_ok": rb_low <= float(row["params_soft_review_budget"]) <= rb_high,
        "corruption_ok": str(row["params_soft_corruption_level"]) in allowed,
    }
    valid = all(checks.values())

    result = row[RAW_DISPLAY_COLUMNS].to_dict()
    result["task_name"] = task_name
    result["shown_in_option_table"] = bool(run_id in set(OPTIONS[task_name]["run_id"]))
    result.update(checks)
    result["same_performance_ok"] = np.nan
    result["review_budget_10pct_lower_ok"] = np.nan
    result["different_from_base_ok"] = np.nan
    result["could_not_find"] = False
    result["valid_selection"] = bool(valid)
    result["n_violations"] = int(sum(not ok for ok in checks.values()))
    result["justification"] = answer.get("justification", "")
    result["confidence"] = validate_confidence(task_name, answer.get("confidence", ""))
    return result


def what_if_alternatives_exist(base_run_id):
    base = df.loc[df["run_id"] == int(base_run_id)].iloc[0]
    min_value = float(base["value"]) - WHAT_IF_TASK["quality_tolerance"]
    max_review = float(base["params_soft_review_budget"]) * WHAT_IF_TASK["review_budget_multiplier"]
    return bool(((df["run_id"] != int(base_run_id)) & (df["value"] >= min_value) & (df["params_soft_review_budget"] <= max_review)).any())


def check_what_if_answer(answer, base_run_id):
    if base_run_id is None:
        raise ValueError("task_3: task_2 base run_id is missing")

    confidence = validate_confidence("task_3", answer.get("confidence", ""))
    could_not_find = bool(answer.get("could_not_find", False))
    alternatives_exist = what_if_alternatives_exist(base_run_id)

    if could_not_find:
        valid = not alternatives_exist
        return {
            "task_name": "task_3",
            "run_id": None,
            "base_run_id": int(base_run_id),
            "shown_in_option_table": False,
            "could_not_find": True,
            "valid_selection": bool(valid),
            "n_violations": 0 if valid else 1,
            "quality_ok": np.nan,
            "train_fraction_ok": np.nan,
            "review_budget_ok": np.nan,
            "corruption_ok": np.nan,
            "same_performance_ok": not alternatives_exist,
            "review_budget_10pct_lower_ok": not alternatives_exist,
            "different_from_base_ok": not alternatives_exist,
            "justification": answer.get("justification", ""),
            "confidence": confidence,
        }

    run_id = answer.get("run_id")
    if run_id is None:
        raise ValueError("task_3: enter a run_id or set could_not_find=True")

    run_id = int(run_id)
    if run_id not in set(df["run_id"]):
        raise ValueError(f"task_3: run_id {run_id} does not exist")

    row = df.loc[df["run_id"] == run_id].iloc[0]
    base = df.loc[df["run_id"] == int(base_run_id)].iloc[0]

    base_value = float(base["value"])
    base_review_budget = float(base["params_soft_review_budget"])
    required_max_review_budget = base_review_budget * WHAT_IF_TASK["review_budget_multiplier"]
    minimum_acceptable_value = base_value - WHAT_IF_TASK["quality_tolerance"]

    checks = {
        "same_performance_ok": float(row["value"]) >= minimum_acceptable_value,
        "review_budget_10pct_lower_ok": float(row["params_soft_review_budget"]) <= required_max_review_budget,
        "different_from_base_ok": run_id != int(base_run_id),
    }
    valid = all(checks.values())

    what_if_table = globals().get("WHAT_IF_OPTIONS", pd.DataFrame({"run_id": []}))
    result = row[RAW_DISPLAY_COLUMNS].to_dict()
    result["task_name"] = "task_3"
    result["base_run_id"] = int(base_run_id)
    result["base_quality_score"] = base_value
    result["base_review_budget"] = base_review_budget
    result["minimum_acceptable_quality"] = minimum_acceptable_value
    result["required_max_review_budget"] = required_max_review_budget
    result["shown_in_option_table"] = bool(run_id in set(what_if_table["run_id"]))
    result["quality_ok"] = np.nan
    result["train_fraction_ok"] = np.nan
    result["review_budget_ok"] = np.nan
    result["corruption_ok"] = np.nan
    result.update(checks)
    result["could_not_find"] = False
    result["valid_selection"] = bool(valid)
    result["n_violations"] = int(sum(not ok for ok in checks.values()))
    result["justification"] = answer.get("justification", "")
    result["confidence"] = confidence
    return result


CHECK_RUN_COUNT = globals().get("CHECK_RUN_COUNT", 0) + 1

records = []
records.append(check_independent_answer("task_1", ANSWERS["task_1"]))
records.append(check_independent_answer("task_2", ANSWERS["task_2"]))
records.append(check_what_if_answer(ANSWERS["task_3"], ANSWERS["task_2"]["run_id"]))

results = pd.DataFrame(records)
summary_columns = [
    "task_name",
    "run_id",
    "value",
    "params_soft_train_fraction",
    "params_soft_review_budget",
    "params_soft_corruption_level",
    "shown_in_option_table",
    "quality_ok",
    "train_fraction_ok",
    "review_budget_ok",
    "corruption_ok",
    "same_performance_ok",
    "review_budget_10pct_lower_ok",
    "different_from_base_ok",
    "could_not_find",
    "valid_selection",
    "confidence",
    "justification",
]
results_display = results[summary_columns].rename(columns=COLUMN_LABELS)
display(results_display)

if results["valid_selection"].all():
    print("All selections satisfy the task constraints.")
else:
    print("At least one selection violates the task constraints. Use the check columns above and choose another answer.")

payload = {
    "elapsed_seconds": round(time.time() - started_at, 2),
    "checker_runs_in_session": CHECK_RUN_COUNT,
    "answers": to_jsonable(records),
    "tasks": to_jsonable(TASKS),
    "what_if_task": to_jsonable(WHAT_IF_TASK),
}

json_path = OUTPUT_DIR / "student_baseline_answers.json"
csv_path = OUTPUT_DIR / "student_baseline_answers.csv"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2)
results.to_csv(csv_path, index=False)

print(f"Saved: {json_path}")
print(f"Saved: {csv_path}")

You are done when the final cell says all selections satisfy the task constraints and the two output files are saved.